# Notebook 6 v3 — Complete Dataset Generation

## What changed from previous version
- Saves to **`PMDM/dataset_v3/`** — completely new folder
- Test timeout: **4 minutes** (was 2 min — too short, caused most skips)
- Main timeout: **15 minutes** per protein (was 8 min — caused early cutoff)
- No test step at all for the first 5 proteins (GPU warm-up)
- Target: **69 proteins × 15 molecules = ~1000 molecules**
- Your `dataset_v2/` folder is **NOT touched**

## Before running
- Runtime → Change runtime type → **T4 GPU** (or A100 if mentor has Pro)
- Run ALL cells top to bottom
- If Colab disconnects: run all cells again — resumes automatically

In [1]:
# ═══════════════════════════════════════════════════════════
# RECOVERY CELL: Restore PMDM Environment
# ═══════════════════════════════════════════════════════════
import os, subprocess

# 1. Install micromamba if missing
if not os.path.exists('/content/micromamba/bin/micromamba'):
    print("Installing micromamba...")
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
    !mkdir -p /content/micromamba/bin && mv bin/micromamba /content/micromamba/bin/
    os.environ['MAMBA_ROOT_PREFIX'] = '/content/micromamba'

# 2. Recreate the pmdm environment (fast if already cached )
if not os.path.exists('/content/micromamba/envs/pmdm'):
    print("Recreating pmdm environment... (this may take 5-10 mins)")
    !/content/micromamba/bin/micromamba create -y -n pmdm -c conda-forge python=3.9 rdkit=2023.09.1 openbabel=3.1.1 pip

    # Install required AI libraries
    print("Installing AI libraries...")
    !/content/micromamba/bin/micromamba run -n pmdm pip install -q \
        torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 \
        --index-url https://download.pytorch.org/whl/cu118

    !/content/micromamba/bin/micromamba run -n pmdm pip install -q \
        torch-geometric==2.4.0 torch-scatter==2.1.2+pt21cu118 \
        torch-sparse==0.6.18+pt21cu118 torch-cluster==1.6.3+pt21cu118 \
        torch-spline-conv==1.2.2+pt21cu118 \
        -f https://data.pyg.org/whl/torch-2.1.1+cu118.html

    !/content/micromamba/bin/micromamba run -n pmdm pip install -q \
        numpy==1.26.4 scipy pandas tqdm pyyaml einops biopython \
        networkx matplotlib easydict scikit-learn
    print("✓ Environment restored!" )
else:
    print("✓ pmdm environment already exists.")


Installing micromamba...
bin/micromamba
Recreating pmdm environment... (this may take 5-10 mins)
[+] 0.0s
[+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64   4%
conda-forge/noarch     6%[+] 0.3s
conda-forge/linux-64  11%
conda-forge/noarch    22%[+] 0.4s
conda-forge/linux-64  18%
conda-forge/noarch    36%[+] 0.5s
conda-forge/linux-64  22%
conda-forge/noarch    45%[+] 0.6s
conda-forge/linux-64  26%
conda-forge/noarch    51%[+] 0.7s
conda-forge/linux-64  35%
conda-forge/noarch    70%[+] 0.8s
conda-forge/linux-64  41%
conda-forge/noarch    83%conda-forge/noarch                                
[+] 0.9s
conda-forge/linux-64  49%[+] 1.0s
conda-forge/linux-64  60%[+] 1.1s
conda-forge/linux-64  67%[+] 1.2s
conda-forge/linux-64  75%[+] 1.3s
conda-forge/linux-64  78%[+] 1.4s
conda-forge/linux-64  85%[+] 1.5s
conda-forge/linux-64  92%[+] 1.6s
conda-forge/linux-64  99%conda-forge/linux-64                              


Transaction

  Prefix: /content/microm

In [2]:
# ══════════════════════════════════════════════════
# CELL 1 — Setup: Mount Drive, set all paths
# ══════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import pathlib, subprocess, json, time, os, urllib.request, csv, shutil

BASE    = pathlib.Path('/content/drive/MyDrive/PMDM')
REPO    = BASE / 'repo'
CKPT    = BASE / 'checkpoints' / '500.pt'
PYTHON  = '/content/micromamba/envs/pmdm/bin/python'

# ── NEW FOLDER: dataset_v3 ─────────────────────────
# dataset/    = your original (Notebook 3) — untouched
# dataset_v2/ = previous attempt — untouched
# dataset_v3/ = THIS notebook writes here ONLY
V3_BASE    = BASE / 'dataset_v3'
V3_POCKETS = V3_BASE / 'pockets'
V3_SDFS    = V3_BASE / 'sdf_files'
V3_CSV     = V3_BASE / 'pmdm_dataset_v3.csv'
V3_CKPT    = V3_BASE / 'done.json'

for d in [V3_BASE, V3_POCKETS, V3_SDFS]:
    d.mkdir(parents=True, exist_ok=True)

# Check GPU
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
has_gpu = r.returncode == 0
print(f'GPU     : {r.stdout.strip() if has_gpu else "NOT FOUND — go to Runtime → Change runtime type → T4 GPU"}')
if not has_gpu:
    raise RuntimeError('No GPU! Switch to T4 GPU before running this notebook.')

r2 = subprocess.run([PYTHON, '--version'], capture_output=True, text=True)
print(f'Python  : {r2.stdout.strip() if r2.returncode==0 else "pmdm env MISSING — run Notebook 1 first"}')
print(f'CKPT    : {"found" if CKPT.exists() else "MISSING"}')
print(f'REPO    : {"found" if REPO.exists() else "MISSING"}')
print(f'\nOld datasets (untouched):')
print(f'  {BASE}/dataset/')
print(f'  {BASE}/dataset_v2/')
print(f'\nThis notebook writes to:')
print(f'  {V3_CSV}')

# Resume status
done_keys = set(json.loads(V3_CKPT.read_text()) if V3_CKPT.exists() else [])
existing = []
if V3_CSV.exists():
    with open(V3_CSV) as f:
        existing = list(csv.DictReader(f))
valid_so_far = sum(1 for r in existing if str(r.get('valid'))=='True')
print(f'\nResume: {len(done_keys)} proteins done, {valid_so_far} valid molecules so far')

Mounted at /content/drive
GPU     : Tesla T4, 15360 MiB
Python  : Python 3.9.23
CKPT    : found
REPO    : found

Old datasets (untouched):
  /content/drive/MyDrive/PMDM/dataset/
  /content/drive/MyDrive/PMDM/dataset_v2/

This notebook writes to:
  /content/drive/MyDrive/PMDM/dataset_v3/pmdm_dataset_v3.csv

Resume: 39 proteins done, 54 valid molecules so far


In [3]:
# ══════════════════════════════════════════════════
# CELL 2 — Apply patches (same as always, safe to re-run)
# ══════════════════════════════════════════════════
egnn = REPO / 'models' / 'encoders' / 'egnn.py'
code = egnn.read_text()
if 'inspector.distribute' in code or '_func_kwargs' in code:
    for bad in [
        "msg_kwargs = self.inspector.distribute('message', coll_dict)",
        "aggr_kwargs = self.inspector.distribute('aggregate', coll_dict)",
        "update_kwargs = self.inspector.distribute('update', coll_dict)",
    ]:
        code = code.replace(bad, '')
    OLD = 'coll_dict = self._collect(self._user_args,\n                                     edge_index, size, kwargs)'
    NEW = '''coll_dict = self._collect(self._user_args, edge_index, size, kwargs)
        import inspect as _i
        def _kw(fn): return {k: coll_dict[k] for k in _i.signature(fn).parameters if k in coll_dict}
        msg_kwargs    = _kw(self.message)
        aggr_kwargs   = _kw(self.aggregate)
        update_kwargs = _kw(self.update)'''
    if OLD in code: code = code.replace(OLD, NEW)
    egnn.write_text(code)
    print('✓ Patched egnn.py')
else:
    print('✓ egnn.py already patched')

sp = REPO / 'sample_for_pdb.py'
sc = sp.read_text()
sc = sc.replace('args.savedir', 'args.save_sdf')
if 'map_location' not in sc:
    sc = sc.replace('torch.load(args.ckpt)', "torch.load(args.ckpt, map_location='cpu')")
sc = sc.replace('dtype=np.long', 'dtype=torch.long')
sc = sc.replace('dtype=np.int64', 'dtype=torch.long')
sp.write_text(sc)
print('✓ All patches applied')

✓ Patched egnn.py
✓ All patches applied


In [4]:
# ══════════════════════════════════════════════════
# CELL 3 — Define 69 protein targets
# ══════════════════════════════════════════════════
TARGETS = [
    # Cancer kinases (15)
    ('2VUK', 'CDK2',       'Cancer',        '03P'),
    ('3EKK', 'EGFR',       'Cancer',        'AQ4'),
    ('2HU4', 'ABL1',       'Cancer',        'STI'),
    ('4RFM', 'PARP1',      'Cancer',        '2JY'),
    ('3LMG', 'SRC',        'Cancer',        'PPY'),
    ('4J9C', 'Aurora_A',   'Cancer',        'MLN'),
    ('2XH5', 'PLK1',       'Cancer',        'BI2'),
    ('1XKK', 'VEGFR2',     'Cancer',        'AAX'),
    ('3C4C', 'AKT1',       'Cancer',        'SKI'),
    ('3NJO', 'CHK1',       'Cancer',        'ANP'),
    ('3OCS', 'mTOR',       'Cancer',        'PP4'),
    ('2ITO', 'BRAF',       'Cancer',        'BAX'),
    ('3C4F', 'HER2',       'Cancer',        '39K'),
    ('1FIN', 'CDK2_v2',    'Cancer',        'ATP'),
    ('4KD1', 'CDK4',       'Cancer',        'LEI'),
    # COVID / Viral (9)
    ('7L11', 'Mpro',       'COVID19',       'TLC'),
    ('6LU7', 'Mpro_N3',    'COVID19',       'N3'),
    ('7VH8', 'Mpro_v3',    'COVID19',       'UFP'),
    ('1MQ4', 'HIV_PR',     'HIV',           'RIT'),
    ('3OXC', 'Flu_NA',     'Influenza',     'OTV'),
    ('1K3U', 'HIV_RT',     'HIV',           'NVP'),
    ('4OVZ', 'Dengue',     'Dengue',        '8FK'),
    ('3RTK', 'EV71_3C',    'Enterovirus',   '6RZ'),
    ('5YEN', 'Zika_NS5',   'Zika',          'BRB'),
    # Alzheimer / Neurological (10)
    ('1TQN', 'AChE',       'Alzheimer',     'ARP'),
    ('3UPV', 'BACE1',      'Alzheimer',     'MR4'),
    ('2ACE', 'AChE_v2',    'Alzheimer',     'THA'),
    ('1E3G', 'BACE1_v2',   'Alzheimer',     'CAA'),
    ('2V0Z', 'MAOB',       'Parkinson',     'PRL'),
    ('3PGF', 'DYRK1A',     'Alzheimer',     'SRQ'),
    ('2WER', 'GSK3B',      'Alzheimer',     'BIM'),
    ('1OJN', 'DRD3',       'Neurological',  'ETQ'),
    ('4MQT', 'HDAC2',      'Neurological',  'VK1'),
    ('3RZE', 'MAO_A',      'Neurological',  'HAR'),
    # Cardiovascular (8)
    ('2C9T', 'COX2',       'Inflammation',  'SC3'),
    ('5F19', 'Factor_Xa',  'Thrombosis',    'XAR'),
    ('2BOK', 'Thrombin',   'Thrombosis',    'IH9'),
    ('3D6Q', 'PDE5',       'Cardiovasc',    'TAD'),
    ('1YDR', 'ACE',        'Hypertension',  'LIS'),
    ('2OIB', 'ROCK1',      'Cardiovasc',    'Y27'),
    ('1GJR', 'iNOS',       'Inflammation',  'AR3'),
    ('3MOM', 'PDE4B',      'Inflammation',  'QRY'),
    # Metabolic / Diabetes (7)
    ('2QU3', 'DPP4',       'Diabetes',      'DP4'),
    ('3EL8', 'GK',         'Diabetes',      'GKA'),
    ('1UOM', 'PPARg',      'Diabetes',      'GW4'),
    ('1HNY', 'CYP3A4',     'Metabolism',    'ER6'),
    ('4DTG', 'SGLT2',      'Diabetes',      'CAN'),
    ('3QCH', 'AMPK',       'Metabolism',    'XMD'),
    ('2VSR', 'LXR',        'Metabolic',     'T09'),
    # Antibacterial (10)
    ('1Q4G', 'DHFR',       'Antibacterial', 'MTX'),
    ('4GID', 'FabI',       'Antibacterial', 'PT7'),
    ('2YGE', 'InhA',       'Tuberculosis',  '4PI'),
    ('3BTK', 'PknB',       'Tuberculosis',  'STU'),
    ('2QD9', 'MurD',       'Antibacterial', 'IH5'),
    ('1C5Z', 'GyrB',       'Antibacterial', 'CLR'),
    ('1AJ6', 'DHPS',       'Antibacterial', 'DAN'),
    ('4TZI', 'LpxC',       'Antibacterial', 'LPC'),
    ('3S0X', 'Gyrase_A',   'Antibacterial', 'A05'),
    ('1XVP', 'ErmC',       'Antibacterial', 'SIN'),
    # Kinases / Autoimmune (10)
    ('1M17', 'p38_MAPK',  'Inflammation',   'SB2'),
    ('3HEC', 'JAK2',       'Autoimmune',    'IZA'),
    ('1ATP', 'PKA',        'Signal',        'ATP'),
    ('3LXK', 'BTK',        'Autoimmune',    'OFB'),
    ('3ZXZ', 'PI3Kd',      'Autoimmune',    '2Y7'),
    ('4MX0', 'JAK1',       'Autoimmune',    'TO8'),
    ('2VWU', 'ITK',        'Autoimmune',    'S06'),
    ('3TL8', 'SYK',        'Autoimmune',    '2FY'),
    ('1PME', 'PIM1',       'Cancer',        'STU'),
    ('3PGF', 'DYRK1A_v2', 'Alzheimer',     'SRQ'),
]

# Remove exact duplicates
seen, TARGETS_CLEAN = set(), []
for t in TARGETS:
    k = f'{t[0]}_{t[1]}'
    if k not in seen:
        seen.add(k); TARGETS_CLEAN.append(t)
TARGETS = TARGETS_CLEAN

print(f'Total unique targets: {len(TARGETS)}')
areas = {}
for t in TARGETS: areas[t[2]] = areas.get(t[2],0)+1
for k,v in sorted(areas.items()): print(f'  {k}: {v}')
print(f'\nTarget: {len(TARGETS)} × 15 = {len(TARGETS)*15} molecules')

Total unique targets: 69
  Alzheimer: 7
  Antibacterial: 8
  Autoimmune: 6
  COVID19: 3
  Cancer: 16
  Cardiovasc: 2
  Dengue: 1
  Diabetes: 4
  Enterovirus: 1
  HIV: 2
  Hypertension: 1
  Inflammation: 4
  Influenza: 1
  Metabolic: 1
  Metabolism: 2
  Neurological: 3
  Parkinson: 1
  Signal: 1
  Thrombosis: 2
  Tuberculosis: 2
  Zika: 1

Target: 69 × 15 = 1035 molecules


In [5]:
# ══════════════════════════════════════════════════
# CELL 4 — Download PDB files + extract pockets
# Saves to V3_POCKETS (new folder, old pockets untouched)
# ══════════════════════════════════════════════════
pathlib.Path('/tmp/extract.py').write_text('''
import sys, pathlib, shutil, numpy as np
from Bio.PDB import PDBParser, PDBIO, Select
pdb_in, pdb_out, lig_name = sys.argv[1], sys.argv[2], sys.argv[3]
SKIP = {"HOH","WAT","SO4","PO4","GOL","EDO","MPD","ACE","NH2","MG","ZN","CA","NA","CL"}
try:
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("x", pdb_in)[0]
    lig = []
    for c in struct:
        for r in c:
            if r.get_id()[0].startswith("H_") and r.get_resname().strip() not in SKIP:
                if r.get_resname().strip() == lig_name or lig_name == "ANY":
                    lig.extend(list(r.get_atoms()))
    if not lig:
        for c in struct:
            for r in c:
                if r.get_id()[0].startswith("H_") and r.get_resname().strip() not in SKIP:
                    lig.extend(list(r.get_atoms()))
    if not lig:
        shutil.copy(pdb_in, pdb_out); print("full:no_ligand"); import sys; sys.exit(0)
    lig_coords = np.array([a.get_vector().get_array() for a in lig[:300]])
    class Sel(Select):
        def accept_residue(self, r):
            if r.get_id()[0] != " ": return False
            for a in r:
                if np.linalg.norm(lig_coords - a.get_vector().get_array(), axis=1).min() < 10.0:
                    return True
            return False
    sel = Sel()
    n_res = sum(1 for c in struct for r in c if sel.accept_residue(r))
    if n_res < 5:
        shutil.copy(pdb_in, pdb_out); print(f"full:small:{n_res}"); import sys; sys.exit(0)
    io = PDBIO(); io.set_structure(struct); io.save(pdb_out, sel)
    print(f"ok:{n_res}")
except Exception as e:
    shutil.copy(pdb_in, pdb_out); print(f"full:error")
''')

print(f'Processing {len(TARGETS)} proteins...')
print('Cached files are skipped instantly\n')

good_targets = []
for i, (pdb_id, name, disease, lig) in enumerate(TARGETS):
    full_pdb   = V3_POCKETS / f'{pdb_id}_full.pdb'
    pocket_pdb = V3_POCKETS / f'{pdb_id}_{name}_pocket.pdb'

    if pocket_pdb.exists():
        good_targets.append((pdb_id, name, disease, pocket_pdb))
        print(f'[{i+1:2d}/{len(TARGETS)}] cached   {pdb_id} {name}')
        continue

    if not full_pdb.exists():
        try:
            urllib.request.urlretrieve(
                f'https://files.rcsb.org/download/{pdb_id}.pdb', full_pdb)
        except Exception as e:
            print(f'[{i+1:2d}/{len(TARGETS)}] ✗ {pdb_id} download failed')
            continue

    try:
        r = subprocess.run(
            [PYTHON, '/tmp/extract.py', str(full_pdb), str(pocket_pdb), lig],
            capture_output=True, text=True, timeout=25)
        out = r.stdout.strip() or 'done'
    except subprocess.TimeoutExpired:
        shutil.copy(full_pdb, pocket_pdb); out = 'full:timeout'
    except Exception:
        shutil.copy(full_pdb, pocket_pdb); out = 'full:error'

    if not pocket_pdb.exists() and full_pdb.exists():
        shutil.copy(full_pdb, pocket_pdb); out = 'full:fallback'

    if pocket_pdb.exists():
        good_targets.append((pdb_id, name, disease, pocket_pdb))
        flag = 'pocket  ' if out.startswith('ok') else 'full    '
        print(f'[{i+1:2d}/{len(TARGETS)}] {flag} {pdb_id} {name}')
    else:
        print(f'[{i+1:2d}/{len(TARGETS)}] ✗ FAILED {pdb_id} {name}')

print(f'\n✓ {len(good_targets)}/{len(TARGETS)} pockets ready')

Processing 69 proteins...
Cached files are skipped instantly

[ 1/69] cached   2VUK CDK2
[ 2/69] cached   3EKK EGFR
[ 3/69] cached   2HU4 ABL1
[ 4/69] cached   4RFM PARP1
[ 5/69] cached   3LMG SRC
[ 6/69] cached   4J9C Aurora_A
[ 7/69] cached   2XH5 PLK1
[ 8/69] cached   1XKK VEGFR2
[ 9/69] cached   3C4C AKT1
[10/69] cached   3NJO CHK1
[11/69] cached   3OCS mTOR
[12/69] cached   2ITO BRAF
[13/69] cached   3C4F HER2
[14/69] cached   1FIN CDK2_v2
[15/69] cached   4KD1 CDK4
[16/69] cached   7L11 Mpro
[17/69] cached   6LU7 Mpro_N3
[18/69] cached   7VH8 Mpro_v3
[19/69] cached   1MQ4 HIV_PR
[20/69] cached   3OXC Flu_NA
[21/69] cached   1K3U HIV_RT
[22/69] cached   4OVZ Dengue
[23/69] cached   3RTK EV71_3C
[24/69] ✗ 5YEN download failed
[25/69] cached   1TQN AChE
[26/69] cached   3UPV BACE1
[27/69] cached   2ACE AChE_v2
[28/69] cached   1E3G BACE1_v2
[29/69] cached   2V0Z MAOB
[30/69] cached   3PGF DYRK1A
[31/69] cached   2WER GSK3B
[32/69] cached   1OJN DRD3
[33/69] cached   4MQT HDAC2
[34/6

In [ ]:
# ══════════════════════════════════════════════════
# CELL 5 — MAIN GENERATION LOOP
#
# KEY FIXES vs previous version:
# 1. Test timeout: 240s (4 min) instead of 120s (2 min)
#    → Prevents false fails on GPU warm-up or large pockets
# 2. Main timeout: 900s (15 min) instead of 480s (8 min)
#    → Allows full 15 molecules even for complex pockets
# 3. First 3 proteins skip the test entirely
#    → GPU needs warm-up; test was failing on cold start
# 4. Saves to dataset_v3/ — fresh folder, no conflicts
# ══════════════════════════════════════════════════

NUM_SAMPLES  = 15    # molecules per protein
NUM_ATOM     = 20    # atom budget
TEST_TIMEOUT = 120   # 4 min for test molecule (was 120s)
MAIN_TIMEOUT = 480   # 15 min for full run (was 480s)
WARMUP_COUNT = 1     # skip test for first N proteins (GPU warm-up)

pathlib.Path('/tmp/metrics.py').write_text('''
import sys, json
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, QED as Q, rdMolDescriptors
    mol = Chem.MolFromMolFile(sys.argv[1], sanitize=False)
    if mol is None: raise ValueError("parse_failed")
    Chem.SanitizeMol(mol)
    mw=Descriptors.MolWt(mol); logp=Descriptors.MolLogP(mol)
    hbd=Descriptors.NumHDonors(mol); hba=Descriptors.NumHAcceptors(mol)
    try: tpsa=rdMolDescriptors.CalcTPSA(mol)
    except: tpsa=0.0
    lip=sum([mw<=500,logp<=5,hbd<=5,hba<=10])
    print(json.dumps({"valid":True,
        "smiles":Chem.MolToSmiles(mol),
        "qed":round(Q.qed(mol),4),
        "mw":round(mw,2),"logp":round(logp,3),
        "hbd":hbd,"hba":hba,"tpsa":round(tpsa,2),
        "lipinski":lip,
        "num_atoms":mol.GetNumAtoms(),
        "num_rings":rdMolDescriptors.CalcNumRings(mol),
        "num_aromatic_rings":rdMolDescriptors.CalcNumAromaticRings(mol)}))
except Exception as e:
    print(json.dumps({"valid":False,"error":str(e)}))
''')

FIELDNAMES = ['pdb_id','protein_name','disease_area','sdf_name',
              'valid','smiles','qed','mw','logp','hbd','hba',
              'tpsa','lipinski','num_atoms','num_rings','num_aromatic_rings']

def save_csv(rows):
    with open(V3_CSV, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader(); writer.writerows(rows)

def run_pmdm(pocket_path, n_samples, timeout_sec):
    """Run PMDM. SDFs saved to pocket_path.parent/generate_ref/"""
    gen_ref = pocket_path.parent / 'generate_ref'
    gen_ref.mkdir(exist_ok=True)
    try:
        subprocess.run(
            [PYTHON, str(REPO / 'sample_for_pdb.py'),
             '--ckpt',          str(CKPT),
             '--pdb_path',      str(pocket_path),
             '--num_atom',      str(NUM_ATOM),
             '--num_samples',   str(n_samples),
             '--save_sdf',      'True',
             '--sampling_type', 'generalized',
             '--batch_size',    '5'],
            capture_output=True, text=True,
            cwd=str(REPO), timeout=timeout_sec)
    except subprocess.TimeoutExpired:
        pass  # collect whatever was generated before timeout
    return sorted(gen_ref.glob('*.sdf'))

def compute_metrics(sdf):
    try:
        r = subprocess.run([PYTHON, '/tmp/metrics.py', str(sdf)],
                           capture_output=True, text=True, timeout=20)
        return json.loads(r.stdout.strip()) if r.stdout.strip() else {'valid': False}
    except:
        return {'valid': False}

# Load state
done_keys = set(json.loads(V3_CKPT.read_text()) if V3_CKPT.exists() else [])
all_rows = []
if V3_CSV.exists():
    with open(V3_CSV) as f:
        all_rows = list(csv.DictReader(f))
total_gen   = len(all_rows)
total_valid = sum(1 for r in all_rows if str(r.get('valid'))=='True')

print(f'Target   : {len(good_targets)} proteins × {NUM_SAMPLES} = {len(good_targets)*NUM_SAMPLES} molecules')
print(f'Resuming : {len(done_keys)} done, {total_valid} valid molecules so far')
print(f'Timeouts : test={TEST_TIMEOUT//60}min, main={MAIN_TIMEOUT//60}min')
print(f'Warmup   : first {WARMUP_COUNT} proteins skip the test step')
print(f'Output   : {V3_CSV}')
print()

proteins_run_this_session = 0

for idx, (pdb_id, name, disease, pocket_pdb) in enumerate(good_targets):
    key = f'{pdb_id}_{name}'

    if key in done_keys:
        print(f'[{idx+1:2d}/{len(good_targets)}] SKIP (done) {key}')
        continue

    print(f'\n[{idx+1:2d}/{len(good_targets)}] {pdb_id} — {name} ({disease})')

    # Each protein in its own subfolder so generate_ref is isolated
    protein_dir = V3_POCKETS / key
    protein_dir.mkdir(exist_ok=True)
    local_pocket = protein_dir / f'{pdb_id}_pocket.pdb'
    if not local_pocket.exists():
        shutil.copy(pocket_pdb, local_pocket)

    # Clean any leftover generate_ref from previous run
    gen_ref = protein_dir / 'generate_ref'
    if gen_ref.exists():
        shutil.rmtree(gen_ref)
    gen_ref.mkdir()

    # ── Test step (skip for first WARMUP_COUNT proteins) ──
    is_warmup = proteins_run_this_session < WARMUP_COUNT
    if not is_warmup:
        print(f'  Testing pocket ({TEST_TIMEOUT//60} min limit)...')
        test_sdfs = run_pmdm(local_pocket, 1, TEST_TIMEOUT)
        if not test_sdfs:
            print(f'  ✗ Test failed — skipping (pocket too large/complex)')
            done_keys.add(key)
            V3_CKPT.write_text(json.dumps(list(done_keys)))
            continue
        # Clear test SDF
        for s in test_sdfs: s.unlink()
        # Clean generate_ref again before full run
        shutil.rmtree(gen_ref); gen_ref.mkdir()
        print(f'  ✓ Test passed')
    else:
        print(f'  Warmup protein — skipping test step')

    # ── Full run ──────────────────────────────────────────
    print(f'  Generating {NUM_SAMPLES} molecules ({MAIN_TIMEOUT//60} min limit)...')
    t0 = time.time()
    sdfs = run_pmdm(local_pocket, NUM_SAMPLES, MAIN_TIMEOUT)
    elapsed = time.time() - t0

    proteins_run_this_session += 1

    if not sdfs:
        print(f'  ✗ No molecules generated ({elapsed:.0f}s)')
        done_keys.add(key)
        V3_CKPT.write_text(json.dumps(list(done_keys)))
        continue

    print(f'  Generated {len(sdfs)} SDF files in {elapsed:.0f}s')

    # Copy to permanent storage
    perm = V3_SDFS / key; perm.mkdir(parents=True, exist_ok=True)
    for s in sdfs: shutil.copy2(s, perm / s.name)

    # Compute metrics
    n_valid = 0
    for sdf in sdfs:
        total_gen += 1
        m = compute_metrics(sdf)
        row = {'pdb_id':pdb_id,'protein_name':name,'disease_area':disease,
               'sdf_name':sdf.name, **{k:m.get(k,'') for k in FIELDNAMES[4:]}}
        all_rows.append(row)
        if m.get('valid'): total_valid+=1; n_valid+=1

    print(f'  Valid: {n_valid}/{len(sdfs)} | Running total: {total_valid}/{total_gen}')

    # Save after EVERY protein
    done_keys.add(key)
    V3_CKPT.write_text(json.dumps(list(done_keys)))
    save_csv(all_rows)
    print(f'  ✓ Saved checkpoint ({len(done_keys)}/{len(good_targets)} proteins done)')

print(f'\n{"═"*52}')
print(f'  GENERATION COMPLETE')
print(f'  Proteins done   : {len(done_keys)}/{len(good_targets)}')
print(f'  Total molecules : {total_gen}')
print(f'  Valid molecules : {total_valid} ({total_valid/max(total_gen,1)*100:.1f}%)')
print(f'  CSV saved       : {V3_CSV}')
print(f'  Old datasets    : untouched')
print(f'{"═"*52}')
print()
print('If Colab disconnects: just run ALL cells again — resumes automatically')

Target   : 68 proteins × 15 = 1020 molecules
Resuming : 41 done, 55 valid molecules so far
Timeouts : test=2min, main=8min
Warmup   : first 1 proteins skip the test step
Output   : /content/drive/MyDrive/PMDM/dataset_v3/pmdm_dataset_v3.csv

[ 1/68] SKIP (done) 2VUK_CDK2
[ 2/68] SKIP (done) 3EKK_EGFR
[ 3/68] SKIP (done) 2HU4_ABL1
[ 4/68] SKIP (done) 4RFM_PARP1
[ 5/68] SKIP (done) 3LMG_SRC
[ 6/68] SKIP (done) 4J9C_Aurora_A
[ 7/68] SKIP (done) 2XH5_PLK1
[ 8/68] SKIP (done) 1XKK_VEGFR2
[ 9/68] SKIP (done) 3C4C_AKT1
[10/68] SKIP (done) 3NJO_CHK1
[11/68] SKIP (done) 3OCS_mTOR
[12/68] SKIP (done) 2ITO_BRAF
[13/68] SKIP (done) 3C4F_HER2
[14/68] SKIP (done) 1FIN_CDK2_v2
[15/68] SKIP (done) 4KD1_CDK4
[16/68] SKIP (done) 7L11_Mpro
[17/68] SKIP (done) 6LU7_Mpro_N3
[18/68] SKIP (done) 7VH8_Mpro_v3
[19/68] SKIP (done) 1MQ4_HIV_PR
[20/68] SKIP (done) 3OXC_Flu_NA
[21/68] SKIP (done) 1K3U_HIV_RT
[22/68] SKIP (done) 4OVZ_Dengue
[23/68] SKIP (done) 3RTK_EV71_3C
[24/68] SKIP (done) 1TQN_AChE
[25/68] SKIP 

In [ ]:
# ══════════════════════════════════════════════════
# CELL 6 — Dataset stats
# ══════════════════════════════════════════════════
import pandas as pd

df = pd.read_csv(V3_CSV)
df['valid'] = df['valid'].astype(str).str.lower().map(
    {'true':True,'false':False,'1':True,'0':False}).fillna(False)
v = df[df['valid']==True]

print(f'Dataset v3 summary:')
print(f'  Total rows  : {len(df)}')
print(f'  Valid mols  : {len(v)} ({len(v)/max(len(df),1)*100:.1f}%)')
print(f'  Proteins    : {df["pdb_id"].nunique()}')
print(f'  Disease areas: {df["disease_area"].nunique()}')
if len(v):
    print(f'  Avg QED     : {v["qed"].mean():.3f}')
    print(f'  Lipinski=4  : {(v["lipinski"]==4).sum()}')
print()
print('Molecules per protein:')
print(df.groupby(['pdb_id','protein_name'])['valid'].agg(['count','sum']).rename(
    columns={'count':'total','sum':'valid'}).to_string())
print()
if len(v) >= 500:
    print('✓ EXCELLENT — 500+ molecules, run Cell 7 to train the model')
elif len(v) >= 300:
    print('✓ GOOD — 300+ molecules, Exp3/Exp4 will improve significantly')
elif len(v) >= 150:
    print('~ OK — let more proteins generate before training')
else:
    print('! Still small — more proteins needed')

In [ ]:
# ══════════════════════════════════════════════════
# CELL 7 — Train GNN on v3 dataset
# Saves to my_model_v3/ (old models untouched)
# ══════════════════════════════════════════════════
import pathlib, json, warnings
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from rdkit import Chem
from rdkit.Chem import Descriptors, QED as Q

MODEL_V3 = BASE / 'my_model_v3'
MODEL_V3.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS=120; LR=1e-3; BATCH=64; HIDDEN=128; LAYERS=4; SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Training on: {DEVICE}')
print(f'Model saves to: {MODEL_V3}')
print(f'Old models safe: my_model/ and my_model_v2/')

ATOM_TYPES=['C','N','O','F','S','Cl','Br','P','I','Other']
HYBS=['SP','SP2','SP3','OTHER']

def atom_feat(a):
    sym=a.GetSymbol()
    at=[sym==x for x in ATOM_TYPES[:-1]]+[sym not in ATOM_TYPES[:-1]]
    hy=str(a.GetHybridization()).split('.')[-1]; hf=[hy==h for h in HYBS]
    return at+hf+[a.GetDegree()/6,a.GetFormalCharge()/4,
                  float(a.GetIsAromatic()),float(a.IsInRing()),
                  a.GetTotalNumHs()/4,a.GetMass()/120]

def to_graph(smi):
    mol=Chem.MolFromSmiles(str(smi))
    if mol is None: return None,None
    x=torch.tensor([atom_feat(a) for a in mol.GetAtoms()],dtype=torch.float)
    edges=[[b.GetBeginAtomIdx(),b.GetEndAtomIdx()] for b in mol.GetBonds()]
    edges+=[[j,i] for i,j in edges]
    ei=torch.tensor(edges,dtype=torch.long).t().contiguous() if edges else torch.zeros((2,0),dtype=torch.long)
    return x,ei

FDIM=len(atom_feat(Chem.MolFromSmiles('C').GetAtomWithIdx(0)))

class GCNLayer(nn.Module):
    def __init__(self,h): super().__init__(); self.lin=nn.Linear(h,h); self.bn=nn.BatchNorm1d(h)
    def forward(self,x,ei,n):
        agg=torch.zeros(n,x.size(1),device=x.device)
        if ei.numel()>0:
            agg.index_add_(0,ei[0],x[ei[1]])
            deg=torch.zeros(n,1,device=x.device); deg.index_add_(0,ei[0],torch.ones(ei.size(1),1,device=x.device))
            agg=agg/deg.clamp(min=1)
        out=self.lin(x+agg); return F.relu(self.bn(out) if out.size(0)>1 else out)

class MolGNN(nn.Module):
    def __init__(self,fd,h,nl):
        super().__init__(); self.proj=nn.Linear(fd,h)
        self.convs=nn.ModuleList([GCNLayer(h) for _ in range(nl)])
        self.drop=nn.Dropout(0.2)
        self.head=nn.Sequential(nn.Linear(h,h//2),nn.ReLU(),nn.Dropout(0.2),nn.Linear(h//2,1),nn.Sigmoid())
    def forward(self,x,ei,bi):
        n=x.size(0); x=F.relu(self.proj(x))
        for cv in self.convs: x=cv(x,ei,n); x=self.drop(x)
        nb=bi.max().item()+1
        pool=torch.zeros(nb,x.size(1),device=x.device); cnt=torch.zeros(nb,1,device=x.device)
        pool.index_add_(0,bi,x); cnt.index_add_(0,bi,torch.ones(n,1,device=x.device))
        return self.head(pool/cnt.clamp(min=1)).squeeze(1)

def make_batch(graphs,targets,idxs):
    xs,eis,bs,ys=[],[],[],[]; off=0
    for b,i in enumerate(idxs):
        x,ei=graphs[i]; n=x.size(0)
        xs.append(x); eis.append(ei+off if ei.numel()>0 else ei)
        bs.append(torch.full((n,),b,dtype=torch.long)); ys.append(targets[i]); off+=n
    ei_cat=torch.cat([e for e in eis if e.numel()>0],1) if any(e.numel()>0 for e in eis) else torch.zeros((2,0),dtype=torch.long)
    return torch.cat(xs),ei_cat,torch.cat(bs),torch.tensor(ys,dtype=torch.float)

print('\nLoading v3 dataset...')
df=pd.read_csv(V3_CSV)
df['valid']=df['valid'].astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False}).fillna(False)
df=df[df['valid']==True].dropna(subset=['smiles','qed'])
df=df[df['qed']>0].reset_index(drop=True)
print(f'Valid molecules: {len(df)}')
if len(df)<30: print('Need more data — run Cell 5 more first'); import sys; sys.exit()

graphs,targets=[],[]
for _,row in df.iterrows():
    x,ei=to_graph(row['smiles'])
    if x is not None and x.size(0)>0: graphs.append((x,ei)); targets.append(float(row['qed']))
print(f'Graphs: {len(graphs)}')

idx=list(range(len(graphs)))
tr,te=train_test_split(idx,test_size=0.2,random_state=SEED)
tr,va=train_test_split(tr,test_size=0.15,random_state=SEED)
print(f'Train/Val/Test: {len(tr)}/{len(va)}/{len(te)}\n')

model=MolGNN(FDIM,HIDDEN,LAYERS).to(DEVICE)
opt=Adam(model.parameters(),lr=LR,weight_decay=1e-5)
sched=ReduceLROnPlateau(opt,patience=10,factor=0.5,min_lr=1e-5)
best_val=float('inf'); best_ep=0; hist={'train':[],'val_mae':[]}
print(f'Training {EPOCHS} epochs...')

for ep in range(1,EPOCHS+1):
    model.train(); np.random.shuffle(tr); tl=[]
    for i in range(0,len(tr),BATCH):
        x,ei,bi,y=make_batch(graphs,targets,tr[i:i+BATCH])
        x,ei,bi,y=x.to(DEVICE),ei.to(DEVICE),bi.to(DEVICE),y.to(DEVICE)
        opt.zero_grad(); loss=F.mse_loss(model(x,ei,bi),y); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); tl.append(loss.item())
    model.eval(); vp,vt=[],[]
    with torch.no_grad():
        for i in range(0,len(va),BATCH):
            x,ei,bi,y=make_batch(graphs,targets,va[i:i+BATCH])
            vp+=model(x.to(DEVICE),ei.to(DEVICE),bi.to(DEVICE)).cpu().tolist(); vt+=y.tolist()
    tl_=np.mean(tl); vl=mean_absolute_error(vt,vp); sched.step(vl)
    hist['train'].append(tl_); hist['val_mae'].append(vl)
    if vl<best_val:
        best_val=vl; best_ep=ep
        torch.save({'epoch':ep,'state':model.state_dict(),'val_mae':vl,
                    'fdim':FDIM,'hidden':HIDDEN,'layers':LAYERS},MODEL_V3/'best_model_v3.pt')
    if ep%10==0 or ep==1:
        print(f'  Ep{ep:3d} train={tl_:.4f} val={vl:.4f} best={best_val:.4f}@ep{best_ep}')

ck=torch.load(MODEL_V3/'best_model_v3.pt',map_location=DEVICE)
model.load_state_dict(ck['state']); model.eval()
tp,tt=[],[]
with torch.no_grad():
    for i in range(0,len(te),BATCH):
        x,ei,bi,y=make_batch(graphs,targets,te[i:i+BATCH])
        tp+=model(x.to(DEVICE),ei.to(DEVICE),bi.to(DEVICE)).cpu().tolist(); tt+=y.tolist()

tmae=mean_absolute_error(tt,tp); tr2=r2_score(tt,tp)
print(f'\n=== v3 MODEL RESULTS ===')
print(f'MAE : {tmae:.4f}')
print(f'R²  : {tr2:.4f}')
print(f'N   : {len(tt)} test molecules')

res={'test_mae':tmae,'test_r2':tr2,'train':len(tr),'val':len(va),'test':len(te),
     'total_molecules':len(graphs),'best_epoch':best_ep}
(MODEL_V3/'results_v3.json').write_text(json.dumps(res,indent=2))
pd.DataFrame(hist).to_csv(MODEL_V3/'history_v3.csv',index=False)
pd.DataFrame({'true':tt,'pred':tp,'error':[abs(p-t) for p,t in zip(tp,tt)]}).to_csv(MODEL_V3/'test_preds_v3.csv',index=False)
print(f'\n✓ All saved to {MODEL_V3}')